<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 5 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">批量导入、失败与重试</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">导入历史业务表和新订单，练习响应检查、安全重试与错误处理。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

先用 Stream Load 导入十笔模拟新订单，练习重试与错误处理，再扩展到 10 张 WWI 历史表（701,846 行）。请按顺序运行。

[讲义](course5_batch_and_streaming_ingestion.md) · [课程入口](../README.md)


## 实验范围

仅重建 orders_imported 和 wwi_orders、wwi_order_lines、wwi_customers、wwi_products、wwi_invoices、wwi_invoice_lines、wwi_customer_transactions、wwi_payment_methods、wwi_transaction_types、wwi_delivery_methods。

历史 Parquet 压缩包和模拟 CSV 随仓库提供；首次使用时自动解包并校验。详见[数据说明](../../datasets/README.md)。独立练习另行重建 orders_mapping_practice。课程工具自动配置沙箱的 BE HTTP 地址。本 Lab 练习 Stream Load；其他接入路径的选择与工作方式见讲义。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()


from uuid import uuid4



### 本次先接入十笔新订单

订单号 900001–900010，来源 COURSE_SIMULATION，金额合计 1400.00。先用这一个 CSV 理解请求参数与结果，之后扩展到 WWI 历史表。


## 1. 导入模拟新订单

使用 off_mode 逐批导入。先查看响应中的 Status、加载行数和过滤行数，再查询目标表核对十笔订单与 1400.00 金额。HTTP 请求成功仅是传输层结果，导入是否完成要看返回的事务状态。


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_imported")
ddl = order_ddl("orders_imported")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
label = "dw_l1_" + uuid4().hex
columns = ",".join(ORDER_COLUMNS)
result = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/orders.csv", label, columns)
show_response(result)
expect(result["Status"], "Success")
expect(result["NumberLoadedRows"], 10)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_imported ORDER BY order_id", title="导入的十笔新订单")


## 2. 同一批次用同 label 重试

立即重发相同文件和 label，观察 Doris 识别已有批次，目标表仍为十行。label 在保留期限内标识导入批次；换新 label 会被视为另一批请求。D07 再用业务键和版本处理事件重复。


In [ ]:
retry = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/orders.csv", label, columns)
show_response(retry)
expect(retry["Status"], "Label Already Exists")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])


## 3. 观察坏数据整批拒绝

新 label 导入两行，其中一行金额无法转换。strict_mode=true 且 max_filter_ratio=0；预期整批失败，原来的十行不变。


In [ ]:
rejected = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/malformed_orders.csv",
                           "dw_bad_" + uuid4().hex, columns)
show_response(rejected)
expect(rejected["Status"], "Fail")
expect(rejected["NumberFilteredRows"], 1)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")])
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_imported", title="错误批次拒绝后，原数据保持完整")


## 4. 扩展到十张历史业务表

完成单文件导入后，再用同样的请求方式批量导入历史 Parquet。先按课程清单校验全部文件，再重建表。阅读 Orders 与 OrderLines、Invoices 与 InvoiceLines 的结构，区分业务表与明细表。


In [ ]:
from dw_course.wwi import manifest, parquet_paths, parquet_ddl
paths = parquet_paths()  # 缺文件或校验和不符会在删表之前停止
for name, metadata in manifest()["tables"].items():
    target = "wwi_" + name
    lab.execute("DROP TABLE IF EXISTS " + target)
    ddl = parquet_ddl(name, target)
    show_sql("WWI 历史表：" + name, ddl)
    lab.execute(ddl)
    response = lab.stream_load(target, paths[name], "wwi_" + uuid4().hex, format="parquet")
    show_response(response)
    expect(response["Status"], "Success")
    expect(response["NumberLoadedRows"], metadata["rows"])
    expect(response["NumberFilteredRows"], 0)
    primary = metadata["primary"]
    expect(lab.query(f"SELECT COUNT(*), COUNT(DISTINCT {primary}) FROM {target}"),
           [(metadata["rows"], metadata["rows"])])


### 导入成功后，回答两个业务问题

1. 订单明细关联订单、客户和商品后，行数是否扩大或丢失？
2. 发票与收款是否表示同一种金额？下面按交易类型展示，保留原始记账正负方向。


In [ ]:
expect(lab.query("""
SELECT COUNT(*), SUM(l.Quantity*l.UnitPrice)
FROM wwi_order_lines l
JOIN wwi_orders o ON l.OrderID=o.OrderID
JOIN wwi_customers c ON o.CustomerID=c.CustomerID
JOIN wwi_products p ON l.StockItemID=p.StockItemID
"""), [(231412, "177634276.40")])
lab.sql("""
SELECT t.TransactionTypeName, COUNT(*) AS rows_count,
       SUM(c.TransactionAmount) AS ledger_amount,
       SUM(CASE WHEN c.InvoiceID IS NULL THEN 1 ELSE 0 END) AS no_invoice_link
FROM wwi_customer_transactions c
JOIN wwi_transaction_types t ON c.TransactionTypeID=t.TransactionTypeID
GROUP BY t.TransactionTypeName ORDER BY t.TransactionTypeName
""", title="发票记账与账户收款")
expect(lab.query("SELECT SUM(TransactionAmount), SUM(OutstandingBalance) FROM wwi_customer_transactions"),
       [("267011.44", "267011.44")])
expect(lab.query("SELECT COUNT(*) FROM wwi_invoices WHERE ConfirmedDeliveryTime IS NOT NULL"), [(70426,)])
lab.sql("""
SELECT o.OrderDate, COUNT(DISTINCT o.OrderID) AS orders,
       SUM(l.Quantity*l.UnitPrice) AS order_amount
FROM wwi_orders o JOIN wwi_order_lines l ON o.OrderID=l.OrderID
GROUP BY o.OrderDate ORDER BY o.OrderDate LIMIT 10
""", title="完整历史上的每日订单分析")


## 完成与排查

导入失败时，保留响应并及时查看可信实验环境中的 ErrorURL，找到错误行及原因。若状态为 Publish Timeout，先根据原 label 与事务信息确认最终结果，再决定重试。D06 将进一步建立长期保留原始输入与拒收原因的表。


## 自己动手与验收

查询 WWI 商品销售额排名，确认每条明细只计入一次。再比较订单金额、发票金额与客户账户收款，说明三者的统计对象。

完成本节后，继续 D06 检查新订单的客户引用与质量，再进入 D07 处理支付、退款及配送事件。


## 独立练习

[orders_reordered.csv](../../datasets/orders_reordered.csv) 有两笔新订单，前两列依次为 customer_id、order_id，其余列沿用原 CSV 顺序。独立创建 orders_mapping_practice，通过 HTTP Stream Load 指定正确的 columns，核对订单 901001 属于客户 1、金额 180；订单 901002 属于客户 2、金额 80。

请自己填写 URL、请求头和列映射；密码从现有连接对象读取，不写入 Notebook。本练习仅重建 orders_mapping_practice。

在下一格编写并运行代码，完成后再展开参考解答。空白练习不会被自动判定为完成。


In [ ]:
# 在这里编写你的 SQL 或导入请求。


<details>
<summary>参考解答（完成后再展开）</summary>

```python
import os
import requests
lab.execute("DROP TABLE IF EXISTS orders_mapping_practice")
ddl = order_ddl("orders_mapping_practice")
show_sql("练习表结构", ddl)
lab.execute(ddl)
# 输入前两列是客户号、订单号；表中的字段顺序与此不同。
practice_columns = "customer_id,order_id,order_amount,status,event_version,event_id,event_time,paid_amount,refund_amount,region,data_source"
practice_url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_mapping_practice/_stream_load"
practice_headers = {
    "label": "mapping_" + uuid4().hex, "format": "csv", "column_separator": ",",
    "columns": practice_columns, "strict_mode": "true", "max_filter_ratio": "0",
    "group_commit": "off_mode",
}
with (COURSE_ROOT / "datasets/orders_reordered.csv").open("rb") as payload:
    response = requests.put(practice_url, auth=(lab.user, lab.password),
                            headers=practice_headers, data=payload, timeout=120,
                            allow_redirects=False)
response.raise_for_status()
result = response.json()
show_response(result, title="独立列映射导入结果")
expect(result["Status"], "Success")
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id", title="订单号与客户号映射结果")
expect(lab.query("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id"),
       [(901001,1,"180.00"),(901002,2,"80.00")])
```

</details>
